### RAG PIPELINES -> DATA INGESTION TO VECTOR DB PIPELINE

In [1]:
# Importing necesssary libraries

import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\JOSHUA ASEMANI\AppData\Local\Temp\ipykernel_2980\2641370141.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\JOSHUA ASEMANI\Music\AI_Chatbots\Traditional-RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Reading all the pdf's in the directory
def process_pdf(pdf_directory_path):
    """Process all pdf in a directory"""
    all_documents=[]
    pdf_dir = Path(pdf_directory_path)

    # find all pdf files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in the directory path")

    # processing individual files
    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            document = loader.load()

            # adding extra meta data to each of the DOC Datatype
            for doc in document:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(document)
            print(f" ✔️ loaded {len(document)} pages")

        except Exception as e:
            print(f" x error: {e}")

    print(f"\n Total pdf document pages loaded accross board is {len(all_documents)}")
    return all_documents

# using the process pdf function
all_pdf_documents = process_pdf("../data")



Found 5 PDF files in the directory path

Processing artificial_intelligence_overview.pdf
 ✔️ loaded 1 pages

Processing cloud_computing_architectures.pdf
 ✔️ loaded 1 pages

Processing cybersecurity_fundamentals.pdf
 ✔️ loaded 1 pages

Processing database_systems_and_storage.pdf
 ✔️ loaded 1 pages

Processing quantum_computing_principles.pdf
 ✔️ loaded 1 pages

 Total pdf document pages loaded accross board is 5


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\artificial_intelligence_overview.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'artificial_intelligence_overview.pdf', 'file_type': 'pdf'}, page_content='Artificial Intelligence: Evolution and Frontiers\nIntroduction\nArtificial Intelligence (AI) represents the simulation of human intelligence processes by machines, especially\ncomputer systems. These processes include learning (the acquisition of information and rules for using the\ninformation), reasoning (using rules to reach approximate or definite conclusions), and self-correction. Over the past\nseveral decades, AI has transitioned from academic curiosity to an essential driver of modern technological\ninfrastructure.\nThe Paradigm Shift from Symbolic to Connectionist AI\nEarly artificial intelligence research focused heavily on symbolic AI, often called Good Old-Fashioned AI (GOFAI).\nSymbolic systems

In [4]:
# Text Splitting gets into chunks -> CHUNKING

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Spliting documents into smaller chucnks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ", ""]
    )

    # using the text splitter object to split the document datatypes we had into another document datatype chunks
    splitted_docs = text_splitter.split_documents(documents)
    print(f"Splitted {len(documents)} document pages into {len(splitted_docs)} chunks")

    # showing an example of a chunnk document type
    print(f"\n Example of a chunk")
    print(f"Content: {splitted_docs[0].page_content[:200]}...")
    print(f"Metadata: {splitted_docs[0].metadata}")

    return splitted_docs

In [5]:
chunks = split_documents(all_pdf_documents)

Splitted 5 document pages into 15 chunks

 Example of a chunk
Content: Artificial Intelligence: Evolution and Frontiers
Introduction
Artificial Intelligence (AI) represents the simulation of human intelligence processes by machines, especially
computer systems. These pro...
Metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\artificial_intelligence_overview.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'artificial_intelligence_overview.pdf', 'file_type': 'pdf'}


### EMBEDDINGS AND VECTOR STORE DB

In [6]:
# Importing necessary libraries
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
# the embedding class
class EmbeddingManager:
    """Handles documents embedding generations using sentence transformers"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: Hugging face model for sentence embeddings
        """
        self.model_name=model_name
        self.model=None
        self._load_model()



    def _load_model(self):
        """Loading the SentenceTransformer models"""
        try:
            print(f"Loading the embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded succesfully. Embedding model dimension: {self.model.get_sentence_embedding_dimension()}")

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise


    def generate_embeddings(self, texts : List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy arrays of embeddings with shape (len(texts), embeddings_dimension)
        """
        if not self.model:
            raise ValueError("Model no Loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated an embeddings with shape {embeddings.shape}")
        return embeddings



    def get_embedding_dimensions(self) -> int:
        """Get the embedding dimension of the model"""
        if not self.model:
            raise ValueError("Model not Loaded")

        return self.model.get_sentence_embedding_dimension()



# Initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading the embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1360.88it/s]


Model Loaded succesfully. Embedding model dimension: 384


C:\Users\JOSHUA ASEMANI\AppData\Local\Temp\ipykernel_2980\4275882136.py:23: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model Loaded succesfully. Embedding model dimension: {self.model.get_sentence_embedding_dimension()}")


### VECTOR STORE CLASS

In [8]:
class VectorStore:
    """Manages the documents embeddings in a chromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the chromaDB collection
            persist_directory: Directory to persist the vector store.
        """
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()




    # Initialize store function
    def _initialize_store(self):
        """Initialize chromaDB client and collection"""

        try:
            # create a persistent chromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or Create Collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise


    # FUnction to add documents chunks and their embeddings
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Adding documents and their embeddings to the vector store

        Args:
            documents: List of langchain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to the vector store")

        # Prepare data for chromaDB
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate Unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)


            # Document content
            documents_text.append(doc.page_content)

            # Embeddings
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,                
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total doucments in collection {self.collection.count()}")


        except Exception as e:
            print(f"Error adding to documents to vector store: {e}")
            raise



# Initializing vector store
vector_store = VectorStore()
vector_store













Vector store initialized. Collection: pdf_documents
Existing documents in collection: 15


In [9]:
# Convert the text to embeddings
chunks
texts = [doc.page_content for doc in chunks]
texts

['Artificial Intelligence: Evolution and Frontiers\nIntroduction\nArtificial Intelligence (AI) represents the simulation of human intelligence processes by machines, especially\ncomputer systems. These processes include learning (the acquisition of information and rules for using the\ninformation), reasoning (using rules to reach approximate or definite conclusions), and self-correction. Over the past\nseveral decades, AI has transitioned from academic curiosity to an essential driver of modern technological\ninfrastructure.\nThe Paradigm Shift from Symbolic to Connectionist AI\nEarly artificial intelligence research focused heavily on symbolic AI, often called Good Old-Fashioned AI (GOFAI).\nSymbolic systems relied upon explicit logic programming, knowledge graphs, and expert decision trees. While\neffective in tightly constrained domains like chess or medical diagnosis rule sets, they suffered from brittleness\nwhen facing ambiguous, noisy real-world data.',
 'effective in tightly co

In [10]:
# Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# store the results in the vector database
vector_store.add_documents(chunks, embeddings)


Generating embeddings for 15 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.96s/it]

Generated an embeddings with shape (15, 384)
Adding 15 documents to the vector store
Successfully added 15 documents to vector store
Total doucments in collection 30


### RAG RETRIEVAL PIPELINE FROM VECTOR STORE

In [19]:
# RAGRetriever class
class RAGRetriever:
    
    """Handles query based retrival from the vector store"""
    def __init__(self, vectorStore: VectorStore, embeddings_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vectorStore: vector store containing documents embeddings
            embeddings_manager: Manager for generating query embeddings
        """

        self.vectorStore = vectorStore
        self.embeddings_manager = embeddings_manager


    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant Documents for a query

        Args: 
            query: the search query
            top_k: Numer of top results to return
            score_threshold: Minimum similiarity score threshold


        Returns:
            List of dictionaries containing retrieved documents and meta data
        """

        print(f"Retrieving Documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate Query embeddings
        query_embedding = self.embeddings_manager.generate_embeddings([query])[0]
        print(query_embedding)

        # search in the vector store
        try:

            results = self.vectorStore.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )

            # printing results
            print(results)
            print(results["documents"])

            # process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i,(doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # convert distances to similarity score (chromaDB uses cosine distance)
                    similarity_score = 1 -  distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })


                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")

            else:
                print("No documents found")

            return retrieved_docs


        except Exception as e:
            print(f"Error during retrieva: {e}")
            return []


# Instansiating the rag retriever class
rag_retriever = RAGRetriever(vector_store, embedding_manager)
rag_retriever



In [20]:
rag_retriever.retrieve("What is artificial intelligence??")

Retrieving Documents for query: 'What is artificial intelligence??'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.22it/s]


Generated an embeddings with shape (1, 384)
[-1.02621056e-02  1.12736737e-03  2.05211435e-02  3.81108262e-02
 -2.37066802e-02 -2.80275457e-02  7.70532936e-02  3.40159889e-03
  1.49544422e-02  5.19539416e-02 -2.20630813e-04  5.42271556e-03
  3.16409357e-02 -2.86062844e-02 -1.16467297e-01  3.34949419e-02
 -2.65850965e-02 -3.10351085e-02 -1.18854903e-01 -7.12215900e-02
  3.27280760e-02  4.31354716e-03 -4.74934317e-02 -3.64126153e-02
 -1.17048677e-02  8.87654945e-02  3.41831637e-03 -3.91336009e-02
 -1.77658850e-03 -7.52476696e-03  3.50280665e-02  1.98026281e-02
  7.66463205e-02  1.52606139e-04 -5.10486625e-02  5.12789376e-02
 -1.14209671e-02  1.67400055e-02  3.52051593e-02 -3.07079460e-02
 -5.04978113e-02 -4.96552140e-02  3.23616751e-02 -1.45999724e-02
  1.02233902e-01  1.48448467e-01 -1.04397781e-01 -3.60445008e-02
 -3.05398442e-02  3.43477353e-02 -1.69256449e-01 -8.27233586e-03
 -5.94484247e-03  1.95294246e-02  4.45652455e-02  3.83730680e-02
  6.04588836e-02 -8.84377770e-03 -3.96968946e-

[{'id': 'doc_080bfe30_0',
  'content': 'Artificial Intelligence: Evolution and Frontiers\nIntroduction\nArtificial Intelligence (AI) represents the simulation of human intelligence processes by machines, especially\ncomputer systems. These processes include learning (the acquisition of information and rules for using the\ninformation), reasoning (using rules to reach approximate or definite conclusions), and self-correction. Over the past\nseveral decades, AI has transitioned from academic curiosity to an essential driver of modern technological\ninfrastructure.\nThe Paradigm Shift from Symbolic to Connectionist AI\nEarly artificial intelligence research focused heavily on symbolic AI, often called Good Old-Fashioned AI (GOFAI).\nSymbolic systems relied upon explicit logic programming, knowledge graphs, and expert decision trees. While\neffective in tightly constrained domains like chess or medical diagnosis rule sets, they suffered from brittleness\nwhen facing ambiguous, noisy real-w

In [23]:
#  Another retrival test
rag_retriever.retrieve("What is Qubits??")

Retrieving Documents for query: 'What is Qubits??'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.94it/s]

Generated an embeddings with shape (1, 384)
[-4.17579189e-02  4.83084396e-02 -1.09314479e-01  1.05057023e-02
 -1.55788690e-01 -2.25916691e-02  6.76246360e-02 -1.64263416e-02
  4.41038460e-02 -9.28026810e-03  1.86496116e-02 -5.49816079e-02
 -1.29142543e-02  1.54091921e-02 -8.89730230e-02 -3.11571993e-02
  3.88014726e-02 -9.43557844e-02  1.77205112e-02 -1.44456653e-03
 -4.46004122e-02  2.75256224e-02 -6.75780252e-02 -1.02044763e-02
  4.70071398e-02  4.42530103e-02 -2.34732013e-02 -3.48775205e-03
  3.49138714e-02 -2.43502855e-02  2.56856158e-02  7.37428665e-02
  6.55063391e-02 -5.62373083e-03 -4.47250530e-02 -2.36563254e-02
  7.60807693e-02 -4.49669920e-03  3.09753586e-02 -5.74008934e-02
 -2.74576358e-02  3.90956178e-03 -1.04217022e-03 -3.26259783e-03
  3.80277261e-02  1.31727114e-01 -1.85039546e-02  4.87605557e-02
  1.17196385e-02 -1.17328964e-01  6.54225238e-03 -2.53774114e-02
  5.20187151e-03  1.58942804e-01  6.12988845e-02  2.61160936e-02
  4.72140610e-02 -4.56610136e-03 -6.58102706e-

[{'id': 'doc_d4b8eae6_14',
  'content': "and Diffie-Hellman public-key schemes.\n- Grover's Algorithm: Provides a quadratic speedup for searching unstructured databases containing N items,\nreducing query complexity from O(N) to O(sqrt(N)).\n- Quantum Simulation: Simulating complex molecular chemistry and condensed-matter physics for battery research\nand drug discovery, a task intractable for classical supercomputers.\nNoisy Intermediate-Scale Quantum (NISQ) and Fault Tolerance\nCurrent quantum hardware operates in the NISQ era, characterized by 50-1000 noisy qubits sensitive to\nenvironmental decoherence and thermal fluctuations. Achieving fault-tolerant quantum computing requires\nQuantum Error Correction (QEC), such as the Surface Code, which weaves thousands of physical qubits into a\nsingle protected logical qubit.",
  'metadata': {'doc_index': 14,
   'total_pages': 1,
   'file_type': 'pdf',
   'creationdate': '',
   'source_file': 'quantum_computing_principles.pdf',
   'page_lab

### LLM INTEGRATION (INTEGRATING VECTOR DB CONTEXT PIPELINE WITH LLM OUTPUT)

In [26]:
#  Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

# Initializing the groq LLM
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key=GROQ_API_KEY, model_name="openai/gpt-oss-120b", temperature=0.1, max_tokens=1024)

# A simple RAG function to retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    # Retrieve the context
    results = retriever.retrieve(query, top_k=top_k)

    context = "\n\n".join([doc["content"] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the questions."

    # generate the answer using GROQ LLM
    prompt = f"""
    use the following context to answer the question concisely.

    context: {context}

    Question: {query}

    Answer:
    """

    response = llm.invoke([prompt.format(context, query)])
    print(response)
    return response.content



In [27]:
answer = rag_simple("What is Qubits", rag_retriever, llm)
answer

Retrieving Documents for query: 'What is Qubits'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 30.61it/s]

Generated an embeddings with shape (1, 384)
[-3.88123468e-02  5.52343689e-02 -1.06516212e-01 -5.58361458e-03
 -1.46461144e-01 -9.95914172e-03  3.86161953e-02 -2.46353652e-02
  4.49324399e-02 -1.43305585e-02 -3.12622823e-03 -6.33511394e-02
 -1.15970429e-02  4.57902960e-02 -1.01222865e-01 -1.58463058e-03
  3.97542827e-02 -7.24494308e-02  5.72934859e-02 -1.58333394e-04
 -6.97608218e-02  2.48077493e-02 -7.73686618e-02 -1.85860824e-02
  5.66403195e-02  3.53018418e-02 -2.04894319e-02 -2.80162394e-02
  1.55569492e-02 -2.50949003e-02  4.84228544e-02  3.84173207e-02
  5.12108058e-02 -1.63444728e-02 -2.68987529e-02 -1.32359257e-02
  8.00698772e-02 -1.50779169e-02  4.54563349e-02 -5.50061874e-02
 -2.47586742e-02  3.22349817e-02 -1.62871722e-02  8.71173851e-03
  2.43190601e-02  1.51746467e-01  3.12499981e-03  5.92284128e-02
  2.09635198e-02 -1.42702177e-01  9.89241153e-03 -2.73565967e-02
  2.65749693e-02  1.61596179e-01  5.50668053e-02  4.50795889e-02
  4.20845784e-02 -3.52689787e-03 -7.39563331e-

content='**Qubits** are the fundamental units of information in a quantum computer.  \n- They are two‑level quantum systems (e.g., spin‑½ particles, photon polarizations) that can exist in a **superposition** of the classical states |0⟩ and |1⟩:\u202f|ψ⟩\u202f=\u202fα|0⟩\u202f+\u202fβ|1⟩ with |α|²\u202f+\u202f|β|²\u202f=\u202f1.  \n- Multiple qubits can become **entangled**, creating correlations that have no classical analogue and enabling quantum algorithms such as Shor’s and Grover’s to achieve speedups.  \n- Operations on qubits are performed with **unitary, reversible quantum gates** (e.g., Pauli‑X, Hadamard, CNOT).  \n\nIn short, a qubit is a quantum analogue of a classical bit, capable of representing and processing information in ways that classical bits cannot.' additional_kwargs={'reasoning_content': 'We need to answer concisely: "What is Qubits". Use context. Provide concise definition. Probably: quantum bits, two-level quantum systems, can be in superposition, entanglement,

'**Qubits** are the fundamental units of information in a quantum computer.  \n- They are two‑level quantum systems (e.g., spin‑½ particles, photon polarizations) that can exist in a **superposition** of the classical states |0⟩ and |1⟩:\u202f|ψ⟩\u202f=\u202fα|0⟩\u202f+\u202fβ|1⟩ with |α|²\u202f+\u202f|β|²\u202f=\u202f1.  \n- Multiple qubits can become **entangled**, creating correlations that have no classical analogue and enabling quantum algorithms such as Shor’s and Grover’s to achieve speedups.  \n- Operations on qubits are performed with **unitary, reversible quantum gates** (e.g., Pauli‑X, Hadamard, CNOT).  \n\nIn short, a qubit is a quantum analogue of a classical bit, capable of representing and processing information in ways that classical bits cannot.'

### ENHANCED RAG PIPELINE FEATURES

In [ ]:
# Enhanced RAG Pipeline features
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
        
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:120] + '...'
    } for doc in results]
    
    confidence = max([doc['similarity_score'] for doc in results])
